# Silver — limpeza e qualidade do Superadmin
Execute após o Bronze, no mesmo catálogo/schema e com Python/PySpark. Preserva nulos opcionais, normaliza códigos e espaços e mantém nomes legíveis. Não preenche datas ou responsáveis fictícios. CPF não existe nas fontes selecionadas e não será consultado.
Duplicatas idênticas são removidas; chaves com conteúdos diferentes ficam em quarentena, sem escolher uma versão arbitrária. As tabelas *_invalidos guardam os motivos e registro_original para revisão. Os snapshots Silver são substituídos a cada execução; o Bronze permanece intacto.
São validados IDs positivos, perfis, categorias, status, referências de usuários, responsável SUPER_ADMIN e cronologia. ATENDIDO exige responsável e encerramento conforme o fluxo de suporte aprovado. Última execução RPA nula é permitida (nenhum sucesso registrado). As regras não inferem se um status intermediário deveria ou não possuir encerramento.
As três fontes Bronze são snapshots separados; execute Bronze por completo antes do Silver. Este notebook não fornece atomicidade entre suas tabelas. Configure a Gold para executar somente após sucesso do Silver, impedindo a publicação de KPIs parciais.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

def texto(coluna):
    # Vazio vira null; nomes preservam acentos e capitalização.
    return F.when(F.trim(F.regexp_replace(F.col(coluna), r"\s+", " ")) != "",
                  F.trim(F.regexp_replace(F.col(coluna), r"\s+", " ")))

def codigo(coluna):
    return F.regexp_replace(
        F.translate(F.upper(texto(coluna)), "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
                    "AAAAAEEEEIIIIOOOOOUUUUC"), r"\s+", "_"
    )

def preparar(tabela, chave):
    fonte = spark.table(tabela)
    campos = [c for c in fonte.columns if c != "ingestado_em"]
    # Ignora só o horário técnico para comparar duplicatas de negócio.
    janela = Window.partitionBy(*campos).orderBy(F.col("ingestado_em").desc_nulls_last())
    unicos = fonte.withColumn("_ordem", F.row_number().over(janela)).filter("_ordem = 1").drop("_ordem")
    unicos = unicos.withColumn("registro_original", F.to_json(F.struct(*[F.col(c) for c in fonte.columns])))
    # IDs iguais com conteúdo diferente não são resolvidos arbitrariamente.
    unicos = unicos.withColumn("_repeticoes", F.count("*").over(Window.partitionBy(chave)))
    return fonte, unicos

def motivos(regras):
    return F.filter(F.array(*[
        F.when(F.coalesce(condicao, F.lit(True)), F.lit(mensagem))
        for condicao, mensagem in regras
    ]), lambda item: item.isNotNull())

def separar(df, regras):
    avaliado = df.withColumn("motivos_invalidade", motivos(regras)).drop("_repeticoes")
    validos = avaliado.filter(F.size("motivos_invalidade") == 0).drop("motivos_invalidade", "registro_original")
    invalidos = avaliado.filter(F.size("motivos_invalidade") > 0)
    return validos, invalidos

def id_invalido(campo):
    return F.col(campo).isNull() | (F.col(campo) <= 0)

resultados = {}


In [ ]:
fonte_u, usuarios = preparar("bronze_site_usuarios", "id")
usuarios = (usuarios
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .withColumn("nome", texto("nome"))
    .withColumn("nivel_acesso", codigo("nivel_acesso"))
)
usuarios_validos, usuarios_invalidos = separar(usuarios, [
    (F.col("_repeticoes") > 1, "ID de usuário duplicado com conteúdo divergente"),
    (id_invalido("id"), "ID de usuário ausente, inválido ou não positivo"),
    (F.col("nome").isNull(), "Nome obrigatório vazio"),
    (~F.col("nivel_acesso").isin("OPERADOR", "GESTOR", "ADMIN", "SUPER_ADMIN"),
     "Nível de acesso ausente ou inválido"),
])
resultados["usuarios"] = (fonte_u, usuarios_validos, usuarios_invalidos)


In [ ]:
fonte_c, chamados = preparar("bronze_site_chamados", "id")
# Nulos opcionais são preservados. Valores preenchidos mas ilegíveis são inválidos.
responsavel_preenchido = texto("cod_usuario_responsavel").isNotNull()
encerramento_preenchido = texto("data_hora_encerramento").isNotNull()
chamados = (chamados
    .withColumn("_responsavel_preenchido", responsavel_preenchido)
    .withColumn("_encerramento_preenchido", encerramento_preenchido)
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .withColumn("cod_usuario_solicitante", F.expr("try_cast(cod_usuario_solicitante AS BIGINT)"))
    .withColumn("cod_usuario_responsavel", F.expr("try_cast(cod_usuario_responsavel AS BIGINT)"))
    .withColumn("data_hora_abertura", F.expr("try_cast(data_hora_abertura AS TIMESTAMP)"))
    .withColumn("data_hora_encerramento", F.expr("try_cast(data_hora_encerramento AS TIMESTAMP)"))
    .withColumn("categoria", codigo("categoria"))
    .withColumn("status", codigo("status"))
)
solicitantes = usuarios_validos.select(F.col("id").alias("_solicitante_existente"))
responsaveis = usuarios_validos.select(
    F.col("id").alias("_responsavel_existente"), F.col("nivel_acesso").alias("_perfil_responsavel"))
chamados = (chamados
    .join(solicitantes, F.col("cod_usuario_solicitante") == F.col("_solicitante_existente"), "left")
    .join(responsaveis, F.col("cod_usuario_responsavel") == F.col("_responsavel_existente"), "left")
)
chamados_validos, chamados_invalidos = separar(chamados, [
    (F.col("_repeticoes") > 1, "ID de chamado duplicado com conteúdo divergente"),
    (id_invalido("id"), "ID de chamado ausente, inválido ou não positivo"),
    (id_invalido("cod_usuario_solicitante") | F.col("_solicitante_existente").isNull(),
     "Solicitante ausente ou não encontrado entre usuários válidos"),
    (~F.col("categoria").isin("DUVIDA", "SUGESTAO", "RECLAMACAO", "OUTRO"),
     "Categoria ausente ou inválida"),
    (~F.col("status").isin("REGISTRADO", "EM_ATENDIMENTO", "ATENDIDO"), "Status ausente ou inválido"),
    (F.col("data_hora_abertura").isNull(), "Data de abertura ausente ou inválida"),
    (F.col("_encerramento_preenchido") & F.col("data_hora_encerramento").isNull(),
     "Data de encerramento preenchida mas inválida"),
    (F.coalesce(F.col("data_hora_encerramento") < F.col("data_hora_abertura"), F.lit(False)),
     "Encerramento anterior à abertura"),
    (F.col("_responsavel_preenchido") &
     (id_invalido("cod_usuario_responsavel") | F.col("_responsavel_existente").isNull() |
      F.coalesce(F.col("_perfil_responsavel") != "SUPER_ADMIN", F.lit(True))),
     "Responsável inválido, inexistente ou sem perfil SUPER_ADMIN"),
    ((F.col("status") == "ATENDIDO") &
     (F.col("data_hora_encerramento").isNull() | F.col("cod_usuario_responsavel").isNull()),
     "Chamado ATENDIDO sem encerramento ou responsável"),
])
tecnicas = ["_responsavel_preenchido", "_encerramento_preenchido",
            "_solicitante_existente", "_responsavel_existente", "_perfil_responsavel"]
chamados_validos = chamados_validos.drop(*tecnicas)
chamados_invalidos = chamados_invalidos.drop(*tecnicas)
resultados["chamados"] = (fonte_c, chamados_validos, chamados_invalidos)


In [ ]:
fonte_r, rpa = preparar("bronze_site_controle_rpa", "nome_carga")
rpa = (rpa
    .withColumn("_data_preenchida", texto("ultima_execucao_sucesso").isNotNull())
    # Nome da carga é identificador técnico: preservar capitalização.
    .withColumn("nome_carga", texto("nome_carga"))
    .withColumn("ultima_execucao_sucesso", F.expr("try_cast(ultima_execucao_sucesso AS TIMESTAMP)"))
)
# Detecta também chaves que ficaram iguais depois de remover espaços.
rpa = rpa.withColumn("_repeticoes", F.count("*").over(Window.partitionBy("nome_carga")))
rpa_validos, rpa_invalidos = separar(rpa, [
    (F.col("_repeticoes") > 1, "Nome da carga duplicado com conteúdo divergente"),
    (F.col("nome_carga").isNull(), "Nome da carga obrigatório vazio"),
    (F.col("_data_preenchida") & F.col("ultima_execucao_sucesso").isNull(),
     "Data de última execução preenchida mas inválida"),
])
# Data nula significa que ainda não houve sucesso: não inventar uma data.
resultados["controle_rpa"] = (fonte_r, rpa_validos.drop("_data_preenchida"),
                            rpa_invalidos.drop("_data_preenchida"))


In [ ]:
resumo = []
for entidade, (fonte, validos, invalidos) in resultados.items():
    # Materializa para que contagens e gravações usem o mesmo resultado.
    validos = validos.cache()
    invalidos = invalidos.cache()
    total = fonte.count()
    quantidade_valida = validos.count()
    quantidade_invalida = invalidos.count()
    duplicatas = total - quantidade_valida - quantidade_invalida
    if duplicatas < 0:
        raise RuntimeError(f"Contagem inconsistente em {entidade}.")
    for sufixo, dados in [(entidade, validos), (f"{entidade}_invalidos", invalidos)]:
        (dados.withColumn("tratado_em", F.current_timestamp())
            .write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true").saveAsTable(f"silver_site_{sufixo}"))
    resumo.append((entidade, total, quantidade_valida, quantidade_invalida, duplicatas))
    validos.unpersist()
    invalidos.unpersist()

qualidade = spark.createDataFrame(
    resumo, "entidade string, total_bronze long, validos long, invalidos long, duplicatas_removidas long"
).withColumn("verificado_em", F.current_timestamp())
qualidade.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_site_qualidade")
display(qualidade)

# Em um Job, a Gold deve depender do sucesso desta etapa.
if any(linha[3] > 0 for linha in resumo):
    raise RuntimeError(
        "Silver gravada com registros em quarentena. Revise silver_site_*_invalidos, "
        "corrija a origem e reexecute Bronze e Silver antes de publicar a Gold."
    )
print("Silver concluída sem registros inválidos.")
